# 32 · Feature engineering sobre E3·S4_s2: inicios de brote

Encargo `docs/modeling/fe_hiperparametros.md`. La arquitectura (E3·S4_s2) y el protocolo
(`protocolo_arquitectura.py`) quedan fijos; este notebook solo orquesta. Nada se escribe en `data/`:
las variables nuevas se calculan en memoria.

## Fase 0 · Preparación y línea base (referencia B)

1. Reproducir E3·S4_s2 con `base_6` y las 5 semillas, **reentrenando** (sin caché del notebook 31), y
   comparar con `fase5_decision.metricas_mediana_semillas` al redondeo de 3 decimales.
2. Guardar la referencia B: medianas y rangos de semillas, antelación y recall en brotes nuevos.
3. Inventario de las candidatas de gold h=2 y sus antecedentes con S1 (ablaciones 25–29).
4. Tiempo de un ajuste completo (5 bloques × 1 semilla).

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.eda.carga import cargar_integrado
from src.modeling import cache_experimentos as cache
from src.modeling import diagnostico_arquitectura as d
from src.modeling import estrategias_horizonte as eh
from src.modeling import fe_hiperparametros as fh
from src.modeling import protocolo_arquitectura as pa
from src.modeling import salida_probabilistica as sp
from src.modeling.dispositivo import configuracion_xgboost
from src.modeling.seguimiento_mlflow import registrar_o_avisar
from src.utils.paths import (DOCS, EXPERIMENTOS_FE_HIPERPARAMETROS, GOLD, GOLD_MANIFEST, METRICAS_ARQUITECTURA,
                             METRICAS_FE_HIPERPARAMETROS, SILVER_INTEGRADO)

warnings.filterwarnings("ignore", message=".*mismatched devices.*")
pd.set_option("display.precision", 3)
HUELLA = cache.huella_encargo()
CACHE = EXPERIMENTOS_FE_HIPERPARAMETROS / "cache"
DESTINO0 = EXPERIMENTOS_FE_HIPERPARAMETROS / "fase0"
print("dispositivo:", configuracion_xgboost(), "| huella:", HUELLA[:12])

panel = cargar_integrado(SILVER_INTEGRADO)
regla = json.loads(GOLD_MANIFEST.read_text(encoding="utf-8"))["regla_brote"]
ins = fh.insumos(panel, regla)
matrices, serie = ins["matrices"], ins["serie"]
pd.DataFrame([{"horizonte": h, "filas": len(m), "causal": bool((m.semana_inicio > m.origen_cierre).all())}
              for h, m in matrices.items()])

dispositivo: {'device': 'cuda', 'n_jobs': 12} | huella: 8c7c6ea89c8c


,horizonte,filas,causal
0,2,30160,True
1,3,30095,True
2,4,30030,True


### 0.3 · Reproducción sin caché

`forzar=True`: siempre reentrena las 5 semillas, aunque exista una caché con la misma huella
(el resultado queda guardado solo para que las fases siguientes lo reutilicen).

In [2]:
referencia = cache.cargar_o_calcular(
    "fase0_referencia_B", lambda: fh.evaluar_variante(matrices, serie), huella=HUELLA, directorio=CACHE, forzar=True)
resumen_b = fh.resumir_variante(referencia["evaluaciones"], serie)
guardada = json.loads(METRICAS_ARQUITECTURA.read_text(encoding="utf-8"))["fase5_decision"]["metricas_mediana_semillas"]
reproduccion = fh.comparar_reproduccion(resumen_b["medianas"], guardada)
print(f"métricas comparadas: {len(reproduccion)} | coinciden a 3 decimales: {int(reproduccion.coincide.sum())}"
      f" | diferencia absoluta máxima: {reproduccion.diferencia_absoluta.max():.2e}")
reproduccion[~reproduccion.coincide]

caché: 'fase0_referencia_B' recalculada (forzada) (huella 8c7c6ea89c8c)


métricas comparadas: 615 | coinciden a 3 decimales: 615 | diferencia absoluta máxima: 4.99e-11


,horizonte,bloque,metrica,guardada,reproducida,diferencia_absoluta,coincide


In [3]:
# Si algo no coincide, el encargo se detiene aquí y se reporta (criterio acordado con Rosa).
if not reproduccion.coincide.all():
    raise RuntimeError("E3·S4_s2 no reproduce fase5_decision al redondeo de 3 decimales")

### 0.4 · Referencia B

Medianas de semillas por horizonte y temporada, con el rango entre semillas (umbral de empate de la regla
simétrica). 2021 es calibración, 2022–2023 selección, 2024 contraste y 2025 sensibilidad de conteos.

In [4]:
COLUMNAS_B = ["horizonte", "bloque", "precision_050", "recall_050", "f1_050", "auprc", "brier",
              "razon_mae", "razon_rmse", "f1_persistencia", "auprc_persistencia"]
medianas_b, rangos_b = resumen_b["medianas"], resumen_b["rangos"]
medianas_b[COLUMNAS_B]

,horizonte,bloque,precision_050,recall_050,f1_050,auprc,brier,razon_mae,razon_rmse,f1_persistencia,auprc_persistencia
0,2,calendario_2025,0.000,0.000,0.000,0.082,0.003,2.016,1.519,0.000,0.010
1,2,temporada_2021,NaN,NaN,NaN,NaN,NaN,1.188,1.054,0.610,0.623
2,2,temporada_2022,0.794,0.601,0.682,0.796,0.051,1.056,1.173,0.686,0.676
3,2,temporada_2023,0.919,0.782,0.845,0.948,0.063,1.005,1.174,0.861,0.899
4,2,temporada_2024,0.750,0.894,0.815,0.894,0.073,1.349,1.780,0.798,0.843
5,3,calendario_2025,0.100,0.333,0.154,0.091,0.004,2.451,1.956,0.154,0.089
6,3,temporada_2021,NaN,NaN,NaN,NaN,NaN,1.198,1.085,0.514,0.502
7,3,temporada_2022,0.869,0.445,0.590,0.784,0.057,1.153,1.377,0.679,0.640
8,3,temporada_2023,0.936,0.716,0.812,0.929,0.077,0.865,0.963,0.829,0.868
9,3,temporada_2024,0.716,0.880,0.789,0.867,0.086,1.465,1.883,0.776,0.803


In [5]:
rangos_b

,horizonte,bloque,f1_050,auprc,brier,razon_mae,rmse,mae,recall_050,precision_050
0,2,calendario_2025,0.000,5.877e-03,1.127e-04,0.014,0.026,0.004,0.000,0.000
1,2,temporada_2021,NaN,NaN,NaN,0.023,0.127,0.012,NaN,NaN
2,2,temporada_2022,0.019,2.894e-03,5.189e-04,0.028,0.476,0.056,0.030,0.013
3,2,temporada_2023,0.004,7.404e-04,3.229e-04,0.028,1.728,0.299,0.006,0.003
4,2,temporada_2024,0.005,9.654e-04,5.265e-04,0.075,2.813,0.370,0.011,0.004
5,3,calendario_2025,0.167,1.474e-02,1.439e-04,0.026,0.047,0.008,0.333,0.111
6,3,temporada_2021,NaN,NaN,NaN,0.011,0.100,0.007,NaN,NaN
7,3,temporada_2022,0.008,3.053e-03,6.976e-04,0.037,0.549,0.089,0.014,0.021
8,3,temporada_2023,0.005,7.533e-04,3.664e-04,0.013,0.600,0.192,0.009,0.004
9,3,temporada_2024,0.004,2.857e-03,9.174e-04,0.049,2.476,0.314,0.015,0.004


Antelación en inicios limpios (fracción de episodios avisados con ≥ 2 semanas): mediana y rango de
semillas, junto a la persistencia. Debajo, la diferencia pareada modelo − persistencia con IC bootstrap
(semilla 17, como en el notebook 31).

In [6]:
resumen_b["antelacion"]

,horizonte,bloque,mediana,minimo,maximo,episodios,persistencia
0,2,seleccion_2022_2023,0.084,0.084,0.095,95,0.116
1,2,temporada_2022,0.053,0.053,0.079,38,0.105
2,2,temporada_2023,0.105,0.105,0.105,57,0.123
3,2,temporada_2024,0.174,0.174,0.174,23,0.130
4,3,seleccion_2022_2023,0.095,0.084,0.105,95,0.137
5,3,temporada_2022,0.053,0.026,0.053,38,0.105
6,3,temporada_2023,0.140,0.123,0.140,57,0.158
7,3,temporada_2024,0.217,0.174,0.304,23,0.087
8,4,seleccion_2022_2023,0.105,0.105,0.116,95,0.211
9,4,temporada_2022,0.000,0.000,0.000,38,0.263


In [7]:
antelacion_pers_b = fh.diferencia_antelacion_persistencia(referencia["evaluaciones"][17])
antelacion_pers_b

,horizonte,episodios,diferencia,ic95_inferior,ic95_superior,diferencia_temporada_2022,diferencia_temporada_2023
0,2,95,-0.032,-0.084,0.021,-0.053,-0.018
1,3,95,-0.042,-0.105,0.011,-0.053,-0.035
2,4,95,-0.105,-0.189,-0.021,-0.263,0.000


Recall en las 2 primeras semanas de brotes nuevos (inicio limpio) frente a semanas de brotes en curso
(alerta p ≥ 0,50 calibrada). Se reporta siempre; no decide.

In [8]:
recall_b = resumen_b["recall_por_tipo"]
recall_b.pivot_table(index=["horizonte", "tipo"], columns="bloque", values="recall_mediana").round(3)

bloque                 calendario_2025  temporada_2022  temporada_2023  \
horizonte tipo                                                           
2         brote_nuevo              0.0           0.061           0.093   
          en_curso                 0.0           0.670           0.844   
3         brote_nuevo              0.0           0.041           0.093   
          en_curso                 1.0           0.500           0.772   
4         brote_nuevo              0.0           0.000           0.081   
          en_curso                 0.0           0.369           0.717   

bloque                 temporada_2024  
horizonte tipo                         
2         brote_nuevo           0.206  
          en_curso              0.922  
3         brote_nuevo           0.147  
          en_curso              0.910  
4         brote_nuevo           0.118  
          en_curso              0.898

In [9]:
cumplimiento_b = sp.cumplimiento_hu0007_4(medianas_b[medianas_b.bloque.isin([*pa.BLOQUES_SELECCION, pa.BLOQUE_CONTRASTE])])
cumplimiento_b

,horizonte,bloque,precision_050,recall_050,f1_050,razon_mae,razon_rmse,cumple_precision_050,cumple_recall_050,cumple_f1_050,cumple_razon_mae,cumple_razon_rmse,cumple_clasificacion,cumple_regresion
2,2,temporada_2022,0.794,0.601,0.682,1.056,1.173,True,False,False,False,False,False,False
3,2,temporada_2023,0.919,0.782,0.845,1.005,1.174,True,False,True,False,False,False,False
4,2,temporada_2024,0.750,0.894,0.815,1.349,1.780,True,True,True,False,False,True,False
7,3,temporada_2022,0.869,0.445,0.590,1.153,1.377,True,False,False,False,False,False,False
8,3,temporada_2023,0.936,0.716,0.812,0.865,0.963,True,False,True,False,False,False,False
9,3,temporada_2024,0.716,0.880,0.789,1.465,1.883,True,True,True,False,False,True,False
12,4,temporada_2022,0.890,0.327,0.478,1.293,1.666,True,False,False,False,False,False,False
13,4,temporada_2023,0.941,0.664,0.779,0.793,0.846,True,False,True,True,True,False,True
14,4,temporada_2024,0.664,0.867,0.751,1.504,1.874,True,True,True,False,False,True,False


In [10]:
DESTINO0.mkdir(parents=True, exist_ok=True)
pd.concat(referencia["predicciones"].values(), ignore_index=True).to_csv(
    DESTINO0 / "predicciones_referencia_B.csv.gz", index=False)
sorted(p.name for p in DESTINO0.iterdir())

['predicciones_referencia_B.csv.gz']

### 0.5 · Candidatas de gold h=2 y antecedentes con S1

Familia y nulos de cada candidata predictora de gold h=2 (`contrato_xgboost.familias_predictoras`).
Las 6 de `base_6` están incluidas en el conteo.

In [11]:
gold_h2 = pd.read_csv(GOLD / f"{SILVER_INTEGRADO.stem}_h2_gold.csv", dtype={"ubigeo": "string"},
                      parse_dates=["semana_inicio"])
inventario = fh.inventario_candidatas(gold_h2, matrices[2])
display(inventario.groupby("familia").agg(columnas=("columna", "size"), nulos_gold_max=("nulos_gold", "max"),
                                          nulos_tras_arranque_max=("nulos_tras_arranque", "max")))
inventario

,columnas,nulos_gold_max,nulos_tras_arranque_max
familia,,,
calendario,2,0,0
clima,2,390,65
historia,4,325,0
jerarquia,4,130,0
socio_2017,4,0,0
socio_anual_reconstruida,16,0,0
vecinos,2,130,0


,columna,familia,en_base_6,nulos_gold,fraccion_nulos_gold,primera_semana_con_valor,nulos_tras_arranque
0,casos_lag_2,historia,True,130,0.004,2017-01-15,0
1,casos_lag_3,historia,True,195,0.006,2017-01-22,0
2,casos_media_4_h2,historia,True,325,0.011,2017-02-05,0
3,casos_semanas_positivas_4_h2,historia,True,325,0.011,2017-02-05,0
4,semana_epi_seno,calendario,True,0,0.000,2017-01-01,0
5,semana_epi_coseno,calendario,True,0,0.000,2017-01-01,0
6,vecinos_media_casos_knn5_h2,vecinos,False,130,0.004,2017-01-15,0
7,vecinos_frac_con_casos_knn5_h2,vecinos,False,130,0.004,2017-01-15,0
8,provincia_otros_casos_h2,jerarquia,False,130,0.004,2017-01-15,0
9,provincia_otros_frac_con_casos_h2,jerarquia,False,130,0.004,2017-01-15,0


**Antecedente, no comparable.** Las ablaciones 25–29 usaron S1 (regresor `log1p` + clasificador) y su propio
protocolo: umbral elegido en validación, F1 medio de 2021–2023 y una semilla, sin rangos de semillas ni regla
simétrica. Las diferencias de F1 frente a `base_6` se listan solo como contexto para la fase 2.

In [12]:
antecedentes = fh.antecedentes_s1(fh.cargar_ablaciones(DOCS / "modeling" / "metricas"))
antecedentes

,ablacion,variante,columnas_agregadas,elegida_en_validacion,f1_val_medio_reg,delta_f1_val_reg_vs_base_6,f1_val_medio_cls,delta_f1_val_cls_vs_base_6
0,25_espacial,base_6,[],NaN,0.677,0.000e+00,0.592,0.000e+00
1,25_espacial,mas_vecinos,"[vecinos_media_casos_knn5_h2, vecinos_frac_con...",NaN,0.680,3.215e-03,0.593,3.686e-04
2,25_espacial,mas_jerarquia,"[provincia_otros_casos_h2, provincia_otros_fra...","regresion, clasificacion",0.688,1.036e-02,0.604,1.184e-02
3,25_espacial,mas_ambos,"[vecinos_media_casos_knn5_h2, vecinos_frac_con...",NaN,0.667,-1.035e-02,0.595,2.362e-03
4,26_clima,base_6,[],"regresion, clasificacion",0.667,0.000e+00,0.592,0.000e+00
5,26_clima,mas_clima_observado,"[hum_rel_media_media5_h2, precip_total_mm_medi...",NaN,0.643,-2.371e-02,0.534,-5.727e-02
6,26_clima,mas_anomalias,"[hum_rel_media_anomalia_media5_h2, precip_tota...",NaN,0.637,-2.992e-02,0.494,-9.716e-02
7,26_clima,mas_ambos,"[hum_rel_media_media5_h2, precip_total_mm_medi...",NaN,0.627,-4.035e-02,0.518,-7.413e-02
8,27_sociodemografica,base_6,[],NaN,0.677,0.000e+00,0.592,0.000e+00
9,27_sociodemografica,fija_poblacion,[log_poblacion_censo_2017],"fija:regresion, fija:clasificacion",0.680,2.579e-03,0.655,6.277e-02


### 0.6 · Tiempo de un ajuste completo

5 bloques × 1 semilla (predicción E3·S4_s2) más la evaluación del protocolo, 3 repeticiones.

In [13]:
tiempos = fh.medir_ajuste(matrices, serie, repeticiones=3)
tiempos.describe().loc[["mean", "min", "max"]]

,repeticion,segundos_ajuste_prediccion,segundos_evaluacion,segundos_total
mean,2.0,4.653,2.623,7.276
min,1.0,4.558,2.619,7.178
max,3.0,4.720,2.629,7.342


### Registro (JSON y MLflow)

In [14]:
reporte_b = fh.reporte_mlflow("fe_hiperparametros_fase0_referencia_B", {fh.NOMBRE_REFERENCIA: medianas_b})
ruta = d.guardar_seccion_metricas(METRICAS_FE_HIPERPARAMETROS, "fase0_referencia", {
    "arquitectura": {"estrategia": fh.ESTRATEGIA, "formulacion": fh.FORMULACION, "columnas": list(eh.GENERICAS),
                     "hiperparametros": reporte_b["parametros"], "semillas": list(pa.SEMILLAS)},
    "reproduccion": {"criterio": "igualdad al redondeo de 3 decimales con fase5_decision.metricas_mediana_semillas",
                     "metricas_comparadas": int(len(reproduccion)), "coinciden": int(reproduccion.coincide.sum()),
                     "diferencia_absoluta_maxima": float(reproduccion.diferencia_absoluta.max()),
                     "reentrenada_sin_cache": True},
    "referencia_B": {"metricas_mediana_semillas": d.a_json(medianas_b), "rangos_semillas": d.a_json(rangos_b),
                     "antelacion": d.a_json(resumen_b["antelacion"]),
                     "antelacion_frente_a_persistencia_semilla_17": d.a_json(antelacion_pers_b),
                     "recall_por_tipo_semana": d.a_json(recall_b),
                     "cumplimiento_hu0007_4": d.a_json(cumplimiento_b)},
    "inventario_candidatas_gold_h2": d.a_json(inventario),
    "antecedentes_s1_ablaciones_25_29": {
        "nota": "S1 y protocolo propio (umbral elegido en validación, F1 medio 2021–2023, una semilla); "
                "antecedente, no comparable con los veredictos de este encargo",
        "variantes": d.a_json(antecedentes)},
    "tiempos_ajuste_completo_segundos": d.a_json(tiempos),
})
print(ruta.relative_to(ROOT))

docs\modeling\metricas\fe_hiperparametros.json


In [15]:
registrar_o_avisar("fe_hiperparametros_fase0_referencia_B", reporte_b, metricas_path=METRICAS_FE_HIPERPARAMETROS,
                   salida_dir=DESTINO0, gold_dir=None, huella=HUELLA,
                   descripcion="Referencia B: E3·S4_s2 con base_6 e hiperparámetros fijos, mediana de 5 semillas")

MLflow: run f6bd74fea8cb4ab0807b43823e203e5a en 'denguekay/fe_hiperparametros_fase0_referencia_B' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


'f6bd74fea8cb4ab0807b43823e203e5a'

## Fase 1 · Diagnóstico de inicios y preregistro de variantes

**Partición A (decidida por Rosa):** las variables se diseñan solo con los inicios limpios de las temporadas
2017–2021. 2022–2023 (selección) y 2024 (contraste) no se miran para diseñar. El diagnóstico con el modelo se
limita a 2021, que es el único bloque de la referencia B dentro del periodo de diseño. La verificación del pico (1.8)
usa 2022–2023, pero solo comprueba un hallazgo del encargo anterior y no mira predictores.

Es descriptivo: diferencias medias inicio − control con IC95 bootstrap por episodio, sin p-valores ni lenguaje
causal. La climatología, el ranking de distritos núcleo y los terciles se ajustan con 2017–2021 **solo para este
diagnóstico**; en la fase 2 se recalculan con el entrenamiento de cada bloque. La regla de consistencia
(«observada», «hipótesis solo 2017», «no observada», «no evaluable») está fijada en el docstring de
`src/modeling/diagnostico_inicios.py`.

In [16]:
from src.modeling import diagnostico_inicios as di
from src.utils.paths import FIGURAS_FE_HIPERPARAMETROS

corte = di.corte_diseno(serie)
inicios = di.inicios_limpios(serie)
diseno = inicios[inicios.verificable & inicios.temporada.isin(di.TEMPORADAS_DISENO)].reset_index(drop=True)
assert diseno.temporada.max() <= 2021 and diseno.inicio.max() <= corte
print("corte del periodo de diseño:", corte.date(), "| inicios verificables de diseño:", len(diseno))

corte del periodo de diseño: 2021-08-28 | inicios verificables de diseño: 71


### 1.1 · Partición: inicios limpios por temporada

«Verificable»: las 8 semanas previas al inicio están dentro del panel (en enero–febrero de 2017 no se puede
comprobar que el inicio sea limpio).

In [17]:
conteo = di.conteo_inicios(inicios)
conteo

,periodo,temporada,limpios,verificables,distritos,semanas_brote_mediana
0,2017,2017,53,40,44,4.0
1,2018_2021,2018,12,12,11,1.0
2,2018_2021,2019,1,1,1,1.0
3,2018_2021,2020,2,2,2,1.0
4,2018_2021,2021,16,16,16,3.5
5,fuera_diseno,2024,23,23,20,4.0
6,fuera_diseno,2025,13,13,13,3.0
7,fuera_diseno,2026,1,1,1,1.0
8,seleccion_2022_2023,2022,38,38,30,2.0
9,seleccion_2022_2023,2023,57,57,49,12.0


### 1.2 · Controles

Ambos tipos usan la misma semana epidemiológica, semanas del periodo de diseño sin brote en la semana ni en las 8
previas y con esas 8 semanas observadas. `semana`: cualquier distrito; `distrito`: el mismo distrito en otras
temporadas (respaldo: mismo tercil de casos 2017–2021 si tiene menos de 2).

In [18]:
elegibles = di.semanas_elegibles(serie)
tercil = di.terciles(serie, corte)
pares = di.controles(diseno, elegibles, tercil)
balance = (pares.groupby(["tipo", "tipo_efectivo", "id_inicio"]).size().rename("controles").reset_index()
           .groupby(["tipo", "tipo_efectivo"]).controles.describe()[["count", "min", "50%", "max"]])
balance

,,count,min,50%,max
tipo,tipo_efectivo,,,,
distrito,distrito,71.0,2.0,4.0,4.0
semana,semana,71.0,232.0,283.0,314.0


In [19]:
estado = di.estado_semanal(panel, serie, corte)
VARIABLES = [v for vs in di.FAMILIAS.values() for v in vs]
tabla_perfiles = di.perfiles(estado, diseno, pares, VARIABLES)
por_rezago = di.resumen_por_rezago(tabla_perfiles)
ventanas = di.por_ventana(tabla_perfiles)
resumen_ventanas = di.resumen_ventanas(ventanas)
veredictos = di.consistencia(resumen_ventanas)
print("pares inicio-variable-k:", len(tabla_perfiles), "| faltantes por k:",
      tabla_perfiles.groupby("k").diferencia.apply(lambda x: round(float(x.isna().mean()), 3)).to_dict())

pares inicio-variable-k: 56232 | faltantes por k: {1: 0.017, 2: 0.017, 3: 0.017, 4: 0.017, 5: 0.026, 6: 0.032, 7: 0.036, 8: 0.052, 9: 0.114, 10: 0.143, 11: 0.154, 12: 0.213}


In [20]:
def mostrar(familia):
    t = veredictos[veredictos.familia.eq(familia)].copy()
    t["variable"] = t.variable.map(di.ETIQUETAS)
    return t.drop(columns="familia").round(3)

def figura(familia, titulo):
    vs = di.FAMILIAS[familia]
    return d.guardar_figura(di.figura_perfiles(por_rezago[por_rezago.periodo_resumen.ne("2017_2021")], vs,
                                               titulo, di.ETIQUETAS), FIGURAS_FE_HIPERPARAMETROS, f"perfil_previo_{familia}")

### 1.3 · Dinámica espacial: vecinos knn5, provincia y región

Nivel y cambio (1 y 4 semanas) de los casos de los otros distritos, de 1 a 12 semanas antes del inicio. Las
ventanas a priori son `origen_2_4` (lo que ven los orígenes de h=2, 3 y 4) y `previa_5_8`.

In [21]:
figura("vecinos", "Vecinos knn5 antes del inicio: diferencia inicio − control (IC95)")
figura("jerarquia", "Provincia y región antes del inicio: diferencia inicio − control (IC95)")
pd.concat([mostrar("vecinos"), mostrar("jerarquia")])

,variable,ventana,veredicto,signo,media_semana_2017,media_semana_2018_2021,media_semana_2017_2021,media_distrito_2017,media_distrito_2018_2021,media_distrito_2017_2021
54,Vecinos knn5: fracción con casos,origen_2_4,observada,1,0.340,0.204,0.280,0.364,0.202,0.294
55,Vecinos knn5: fracción con casos,previa_5_8,observada,1,0.257,0.184,0.225,0.277,0.174,0.232
56,Vecinos knn5: fracción en brote,origen_2_4,observada,1,0.155,0.057,0.112,0.163,0.053,0.115
57,Vecinos knn5: fracción en brote,previa_5_8,observada,1,0.117,0.046,0.086,0.122,0.048,0.090
58,Vecinos knn5: log(1 + casos medios),origen_2_4,observada,1,1.168,0.518,0.884,1.220,0.478,0.896
59,Vecinos knn5: log(1 + casos medios),previa_5_8,observada,1,0.948,0.567,0.782,0.976,0.537,0.784
60,Vecinos: cambio en 1 semana (log),origen_2_4,observada,1,0.081,0.014,0.051,0.093,0.012,0.058
61,Vecinos: cambio en 1 semana (log),previa_5_8,no_observada,0,0.025,-0.072,-0.017,0.029,-0.080,-0.019
62,Vecinos: cambio en 4 semanas (log),origen_2_4,no_observada,0,0.226,-0.087,0.089,0.252,-0.095,0.100
63,Vecinos: cambio en 4 semanas (log),previa_5_8,no_observada,0,0.108,-0.238,-0.055,0.131,-0.270,-0.058


### 1.4 · Distritos núcleo

Los 5 distritos con más casos acumulados en 2017–2021, excluido el propio. «Ya tenían brote» = fracción del
núcleo en brote. Describe co-ocurrencia; la importación desde el núcleo queda como **hipótesis** y no se le da
lectura causal.

In [22]:
nucleo = di.ranking_nucleo(serie, corte).head(di.N_NUCLEO + 1)
display(nucleo.rename("casos_2017_2021").to_frame().join(panel.drop_duplicates("ubigeo").set_index("ubigeo")[["provincia", "distrito"]]))
figura("nucleo", "Distritos núcleo antes del inicio: diferencia inicio − control (IC95)")
mostrar("nucleo")

,casos_2017_2021,provincia,distrito
ubigeo,,,
200101,13190,Piura,Piura
200104,6904,Piura,Castilla
200115,6717,Piura,Veintiséis de Octubre
200601,6441,Sullana,Sullana
200401,2567,Morropón,Chulucanas
200701,1854,Talara,Pariñas


,variable,ventana,veredicto,signo,media_semana_2017,media_semana_2018_2021,media_semana_2017_2021,media_distrito_2017,media_distrito_2018_2021,media_distrito_2017_2021
36,Núcleo: fracción en brote,origen_2_4,observada,1,0.669,0.110,0.425,0.759,0.127,0.483
37,Núcleo: fracción en brote,previa_5_8,observada,1,0.659,0.098,0.414,0.748,0.117,0.472
38,"Núcleo (5 distritos, sin el propio): log(1 + c...",origen_2_4,observada,1,3.955,1.732,2.985,4.492,2.175,3.480
39,"Núcleo (5 distritos, sin el propio): log(1 + c...",previa_5_8,observada,1,3.499,1.673,2.702,3.971,2.112,3.159
40,Núcleo: cambio en 4 semanas (log),origen_2_4,observada,1,0.522,0.044,0.313,0.594,0.042,0.353
41,Núcleo: cambio en 4 semanas (log),previa_5_8,hipotesis_solo_2017,0,0.390,-0.037,0.189,0.439,-0.072,0.199


### 1.5 · Clima con rezagos de 4 a 12 semanas

Temperatura media y mínima, precipitación y humedad relativa semanales, y sus anomalías respecto de la
climatología distrito × semana de 2017–2021. Con el control `distrito` (mismo distrito y semana), la anomalía y el
valor observado dan la misma diferencia, porque la climatología se cancela. Ventanas a priori: 4–8 y 8–12 semanas.

In [23]:
figura("clima", "Clima antes del inicio: diferencia inicio − control (IC95)")
mostrar("clima")

,variable,ventana,veredicto,signo,media_semana_2017,media_semana_2018_2021,media_semana_2017_2021,media_distrito_2017,media_distrito_2018_2021,media_distrito_2017_2021
0,Humedad relativa media (%),clima_4_8,hipotesis_solo_2017,0,5.890,-1.417,2.700,9.640,1.038,5.884
1,Humedad relativa media (%),clima_8_12,hipotesis_solo_2017,0,2.660,-2.260,0.512,6.170,0.603,3.739
2,Anomalía de humedad relativa (puntos %),clima_4_8,observada,1,8.655,0.885,5.262,9.640,1.016,5.874
3,Anomalía de humedad relativa (puntos %),clima_8_12,observada,1,5.654,0.562,3.431,6.170,0.631,3.751
4,Precipitación semanal (mm),clima_4_8,hipotesis_solo_2017,0,42.168,-6.745,20.811,54.393,-4.722,28.582
5,Precipitación semanal (mm),clima_8_12,hipotesis_solo_2017,0,32.007,-8.335,14.393,40.386,-6.146,20.069
6,Anomalía de precipitación (mm),clima_4_8,hipotesis_solo_2017,0,48.893,-2.829,26.310,54.393,-4.740,28.575
7,Anomalía de precipitación (mm),clima_8_12,hipotesis_solo_2017,0,36.110,-3.420,18.850,40.386,-6.139,20.072
8,Temperatura media (°C),clima_4_8,no_observada,0,1.057,1.371,1.194,-0.292,-0.292,-0.292
9,Temperatura media (°C),clima_8_12,no_observada,0,1.240,1.376,1.299,-0.058,-0.348,-0.184


### 1.6 · Historia propia

Semana con casos aislados, semanas desde el último caso, pendiente y aceleración de 4 semanas en escala
log(1 + casos), y casos de la temporada anterior (no existe para 2017).

In [24]:
figura("propia", "Historia propia antes del inicio: diferencia inicio − control (IC95)")
mostrar("propia")

,variable,ventana,veredicto,signo,media_semana_2017,media_semana_2018_2021,media_semana_2017_2021,media_distrito_2017,media_distrito_2018_2021,media_distrito_2017_2021
42,Propio: aceleración (log),origen_2_4,no_observada,1,0.028,0.005,0.018,0.035,0.002,0.021
43,Propio: aceleración (log),previa_5_8,no_observada,1,0.021,0.007,0.015,0.024,0.017,0.021
44,Propio: semana con casos (0/1),origen_2_4,observada,1,0.445,0.291,0.378,0.454,0.269,0.373
45,Propio: semana con casos (0/1),previa_5_8,observada,1,0.362,0.240,0.308,0.363,0.222,0.301
46,Propio: log(1 + casos),origen_2_4,observada,1,0.625,0.405,0.529,0.643,0.396,0.535
47,Propio: log(1 + casos),previa_5_8,observada,1,0.604,0.364,0.499,0.607,0.349,0.494
48,Propio: log(1 + casos de la temporada anterior),origen_2_4,no_evaluable,0,NaN,1.320,1.320,NaN,0.801,0.801
49,Propio: log(1 + casos de la temporada anterior),previa_5_8,no_evaluable,0,NaN,1.320,1.320,NaN,0.801,0.801
50,Propio: pendiente 4 semanas (log),origen_2_4,no_observada,1,0.005,0.016,0.010,0.011,0.019,0.015
51,Propio: pendiente 4 semanas (log),previa_5_8,no_observada,0,0.007,-0.032,-0.010,0.004,-0.036,-0.013


### 1.7 · Inicios de 2021 frente a la referencia B

2021 es el único bloque de la referencia B dentro del periodo de diseño. Es el bloque de calibración inicial, así
que no tiene probabilidad calibrada: se describen la probabilidad sin calibrar y los casos predichos para la
semana de inicio (mediana de las 5 semillas), por horizonte.

In [25]:
prediccion_b = pd.concat(referencia["predicciones"].values(), ignore_index=True)
inicios_2021 = di.inicios_con_prediccion(diseno, prediccion_b)
display(inicios_2021.pivot_table(index=["ubigeo", "inicio", "casos_Dengue"], columns="horizonte",
                                 values=["probabilidad_brote", "casos_predichos", "persistencia"]).round(2))
inicios_2021.groupby("horizonte").agg(inicios=("id_inicio", "size"), prob_max=("probabilidad_brote", "max"),
                                      prob_mediana=("probabilidad_brote", "median"),
                                      con_prob_050=("probabilidad_brote", lambda p: int((p >= 0.5).sum())))

casos_predichos             persistencia       \
horizonte                                    2     3     4            2    3   
ubigeo inicio     casos_Dengue                                                 
200114 2021-07-25 11.0                    6.69  2.73  4.77         10.0  7.0   
200304 2021-06-13 2.0                     0.10  0.10  0.12          0.0  0.0   
200306 2021-05-16 11.0                    0.48  0.12  0.13          1.0  0.0   
200401 2021-01-03 5.0                     1.22  1.79  0.91          0.0  2.0   
200402 2021-07-18 2.0                     0.10  0.35  0.37          0.0  0.0   
200404 2021-08-08 2.0                     0.85  1.14  0.78          0.0  2.0   
200405 2021-05-02 6.0                     0.38  0.45  0.68          0.0  0.0   
200406 2021-08-01 7.0                     1.40  0.97  0.71          2.0  1.0   
200501 2021-07-25 2.0                     0.60  0.77  0.86          0.0  0.0   
200505 2021-04-25 8.0                     0.52  0.13  0.14          1.0  0.0   
200604 2021-07-25 2.0                     0.10  0.10  0.10          0.0  0.0   
200608 2021-08-15 2.0                     0.10  0.10  0.10          0.0  0.0   
200702 2021-03-14 44.0                    0.55  0.12  0.13          1.0  0.0   
200704 2021-05-30 6.0                     0.10  0.12  0.13          0.0  0.0   
200705 2021-06-27 2.0                     1.17  0.73  0.37          2.0  1.0   
200805 2021-08-01 3.0                     0.10  0.10  0.10          0.0  0.0   

                                     probabilidad_brote              
horizonte                          4                  2     3     4  
ubigeo inicio     casos_Dengue                                       
200114 2021-07-25 11.0          12.0               0.24  0.04  0.16  
200304 2021-06-13 2.0            0.0               0.01  0.01  0.01  
200306 2021-05-16 11.0           0.0               0.10  0.01  0.02  
200401 2021-01-03 5.0            0.0               0.32  0.41  0.22  
200402 2021-07-18 2.0            0.0               0.01  0.07  0.08  
200404 2021-08-08 2.0            0.0               0.21  0.28  0.19  
200405 2021-05-02 6.0            1.0               0.00  0.00  0.01  
200406 2021-08-01 7.0            2.0               0.03  0.01  0.01  
200501 2021-07-25 2.0            1.0               0.14  0.19  0.21  
200505 2021-04-25 8.0            0.0               0.03  0.00  0.00  
200604 2021-07-25 2.0            0.0               0.01  0.01  0.01  
200608 2021-08-15 2.0            0.0               0.01  0.01  0.01  
200702 2021-03-14 44.0           0.0               0.12  0.01  0.02  
200704 2021-05-30 6.0            0.0               0.01  0.01  0.02  
200705 2021-06-27 2.0            0.0               0.31  0.18  0.08  
200805 2021-08-01 3.0            0.0               0.01  0.01  0.01

,inicios,prob_max,prob_mediana,con_prob_050
horizonte,,,,
2,16,0.319,0.028,0
3,16,0.414,0.011,0
4,16,0.218,0.016,0


### 1.8 · Verificación: ¿el conteo pierde frente a la persistencia en el pico?

Errores por fase de la curva regional de la referencia B (mediana de semillas; la barra de error es el rango entre
semillas), temporadas 2022 y 2023. Solo verifica el hallazgo del encargo de arquitectura; no diseña variables.

In [26]:
errores_fase = di.errores_por_fase(list(referencia["evaluaciones"].values()), pa.BLOQUES_SELECCION)
d.guardar_figura(di.figura_errores_fase(errores_fase), FIGURAS_FE_HIPERPARAMETROS, "errores_por_fase_referencia_B")
errores_fase.pivot_table(index="fase", columns=["bloque", "horizonte"], values="razon_mae").round(3)

bloque    temporada_2022               temporada_2023              
horizonte              2      3      4              2      3      4
fase                                                               
bajada             0.744  0.688  0.710          0.686  0.520  0.450
pico               1.211  1.578  2.037          1.223  1.118  1.088
subida             1.123  1.107  1.023          0.926  0.838  0.809
valle              0.970  0.944  0.875          1.022  0.952  0.906

In [27]:
errores_fase.round(3)

,horizonte,bloque,fase,n,razon_mae,razon_rmse,sesgo_modelo,sesgo_persistencia,suma_predicha_sobre_real,rango_razon_mae,rango_razon_rmse
0,2,temporada_2022,bajada,455.0,0.744,0.682,1.336,2.468,1.318,0.030,0.034
1,2,temporada_2022,pico,585.0,1.211,1.277,2.599,-0.771,1.237,0.034,0.075
2,2,temporada_2022,subida,715.0,1.123,1.364,0.303,-1.250,1.073,0.052,0.123
3,2,temporada_2022,valle,1625.0,0.970,0.872,0.145,0.187,1.163,0.023,0.046
4,2,temporada_2023,bajada,585.0,0.686,0.619,-7.890,15.308,0.726,0.033,0.044
5,2,temporada_2023,pico,390.0,1.223,1.315,-52.270,-12.587,0.475,0.037,0.038
6,2,temporada_2023,subida,455.0,0.926,0.903,-3.601,-10.125,0.883,0.018,0.020
7,2,temporada_2023,valle,1950.0,1.022,0.950,-0.322,-0.502,0.876,0.005,0.011
8,3,temporada_2022,bajada,455.0,0.688,0.713,2.117,3.688,1.504,0.026,0.031
9,3,temporada_2022,pico,585.0,1.578,1.764,5.768,-1.265,1.527,0.082,0.128


### 1.9 · Síntesis y preregistro

Veredictos por variable y ventana (regla del módulo), resumidos por familia. El preregistro de las variantes de la
fase 2 está en `docs/modeling/fe_hiperparametros.md` («Preregistro») y se guarda aquí en el JSON; no se amplía
después de ver resultados sin la aprobación de Rosa.

In [28]:
veredictos.groupby(["familia", "ventana"]).veredicto.value_counts().unstack(fill_value=0)

veredicto             hipotesis_solo_2017  no_evaluable  no_observada  \
familia   ventana                                                       
clima     clima_4_8                     5             0             1   
          clima_8_12                    5             0             1   
jerarquia origen_2_4                    1             0             0   
          previa_5_8                    3             0             0   
nucleo    origen_2_4                    0             0             0   
          previa_5_8                    1             0             0   
propia    origen_2_4                    0             1             2   
          previa_5_8                    0             1             2   
vecinos   origen_2_4                    0             0             1   
          previa_5_8                    0             0             2   

veredicto             observada  
familia   ventana                
clima     clima_4_8           2  
          clima_8_12          2  
jerarquia origen_2_4          9  
          previa_5_8          7  
nucleo    origen_2_4          3  
          previa_5_8          2  
propia    origen_2_4          3  
          previa_5_8          3  
vecinos   origen_2_4          5  
          previa_5_8          4

In [29]:
from src.modeling import preregistro_fe as pr
PREREGISTRO = pr.como_dict()
print("variantes:", len(PREREGISTRO["variantes"]), "| comparaciones previstas:", PREREGISTRO["comparaciones"]["total"])
pd.DataFrame(PREREGISTRO["variantes"])[["id", "bloque", "familia", "fuente", "columnas", "comparar_contra"]]

variantes: 12 | comparaciones previstas: 15


,id,bloque,familia,fuente,columnas,comparar_contra
0,A1,A,vecinos,epidemiologica,"[vecinos_media_casos_knn5_h2, vecinos_frac_con...",[referencia_B]
1,A2,A,jerarquia,epidemiologica,"[provincia_otros_casos_h2, provincia_otros_fra...",[referencia_B]
2,A3,A,clima_gold,meteorologica,"[hum_rel_media_media5_h2, precip_total_mm_medi...",[referencia_B]
3,A4,A,poblacion_2017,sociodemografica,[log_poblacion_censo_2017],[referencia_B]
4,A5,A,poblacion_anual,sociodemografica,[log_poblacion_anual],[referencia_B]
5,A6,A,fracciones_2017,sociodemografica,"[fraccion_rural_2017, fraccion_menores_15_2017...",[referencia_B]
6,A7,A,fracciones_anuales_cp1,sociodemografica,[socio_anual_cp1],[referencia_B]
7,B1,B,dinamica_espacial,epidemiologica,"[vecinos_frac_en_brote, vecinos_n_casos_nuevos...","[referencia_B, base_6+A1+A2]"
8,B2,B,distritos_nucleo,epidemiologica,"[nucleo_log_casos, nucleo_frac_en_brote, nucle...",[referencia_B]
9,B3,B,historia_propia_ampliada,epidemiologica,"[propia_semanas_desde_ultimo_caso, propia_pend...",[referencia_B]


In [30]:
ruta = d.guardar_seccion_metricas(METRICAS_FE_HIPERPARAMETROS, "fase1_diagnostico", {
    "particion": {"opcion": "A", "temporadas_diseno": list(di.TEMPORADAS_DISENO), "corte": str(corte.date()),
                  "periodos": {k: list(v) for k, v in di.PERIODOS.items()},
                  "modelo_solo_2021": True, "pico_2022_2023_solo_verificacion": True},
    "inicios_por_temporada": d.a_json(conteo),
    "controles": {"minimo_distrito": di.MIN_CONTROLES_DISTRITO,
                  "balance": d.a_json(balance.reset_index())},
    "ventanas_a_priori": {k: list(v) for k, v in di.VENTANAS.items()},
    "nucleo_2017_2021": d.a_json(nucleo.rename("casos").reset_index()),
    "resumen_por_rezago": d.a_json(por_rezago),
    "resumen_por_ventana": d.a_json(resumen_ventanas),
    "veredictos": d.a_json(veredictos),
    "inicios_2021_referencia_B": d.a_json(inicios_2021),
    "errores_por_fase_referencia_B": d.a_json(errores_fase),
})
d.guardar_seccion_metricas(METRICAS_FE_HIPERPARAMETROS, "fase1_preregistro", PREREGISTRO)
print(ruta.relative_to(ROOT))

docs\modeling\metricas\fe_hiperparametros.json
